In [ ]:
import sys
import os
sys.path.insert(0, '/app')

from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, concat_ws, sha2, to_json, struct, current_timestamp
from connectors.clickhouse_client import ClickHouseClient
from config.settings import oracle_config, clickhouse_config, spark_config, app_config

ref_date = datetime.now().strftime("%Y-%m-%d")
previous_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

tables_config = [
    {
        "schema": "ADMBI_PRD",
        "table": "TABLE_1",
        "primary_keys": ["ID"],
    },
    {
        "schema": "ADMBI_PRD",
        "table": "TABLE_2",
        "primary_keys": ["CODE", "VERSION"],
    },
]

spark_home = os.environ.get('SPARK_HOME', '/usr/local/spark')
ojdbc_jar = f"{spark_home}/jars/ojdbc8-21.9.0.0.jar"
clickhouse_jar = f"{spark_home}/jars/clickhouse-jdbc-0.4.6-all.jar"

spark = (
    SparkSession.builder
    .appName(f"FullPipeline_{ref_date}")
    .config("spark.driver.memory", spark_config.driver_memory)
    .config("spark.executor.memory", spark_config.executor_memory)
    .config("spark.executor.cores", spark_config.executor_cores)
    .config("spark.sql.shuffle.partitions", spark_config.sql_shuffle_partitions)
    .config("spark.sql.adaptive.enabled", spark_config.sql_adaptive_enabled)
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.jars", f"{ojdbc_jar},{clickhouse_jar}")
    .getOrCreate()
)

oracle_jdbc_url = f"jdbc:oracle:thin:@//{oracle_config.host}:{oracle_config.port}/{oracle_config.service}"
protocol = "https" if clickhouse_config.secure else "http"
clickhouse_jdbc_url = (
    f"jdbc:clickhouse://{protocol}://{clickhouse_config.host}:"
    f"{clickhouse_config.port}/{clickhouse_config.database}"
)


def ingest_bronze(config):
    schema_name = config["schema"]
    table_name = config["table"]
    primary_keys = config["primary_keys"]

    df_oracle = (
        spark.read
        .format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("dbtable", f"{schema_name}.{table_name}")
        .option("user", oracle_config.user)
        .option("password", oracle_config.password)
        .option("driver", "oracle.jdbc.driver.OracleDriver")
        .option("fetchsize", app_config.batch_size)
        .option("numPartitions", "10")
        .load()
    )

    primary_key_str = concat_ws("|||", *[col(pk) for pk in primary_keys])
    all_columns = [col(c) for c in df_oracle.columns]
    row_data = to_json(struct(*all_columns))
    row_hash_input = concat_ws("|||", *all_columns)

    df_bronze = df_oracle.select(
        lit(ref_date).cast("date").alias("ref_date"),
        lit(table_name).alias("table_name"),
        primary_key_str.alias("primary_key"),
        sha2(row_hash_input, 256).alias("row_hash"),
        row_data.alias("data"),
        current_timestamp().alias("ingestion_timestamp"),
    )

    (
        df_bronze.write
        .format("jdbc")
        .option("url", clickhouse_jdbc_url)
        .option("dbtable", "bronze.snapshot_raw")
        .option("user", clickhouse_config.user)
        .option("password", clickhouse_config.password)
        .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
        .option("batchsize", app_config.batch_size)
        .mode("append")
        .save()
    )

    return table_name


for config in tables_config:
    ingest_bronze(config)

spark.stop()

client = ClickHouseClient()


def process_silver(table_name):
    client.execute_query(
        f"""
        INSERT INTO silver.delta_events (
            ref_date, table_name, primary_key, operation_type, row_hash_after, data_after
        )
        SELECT
            toDate('{ref_date}'),
            '{table_name}',
            current.primary_key,
            'INSERT',
            current.row_hash,
            current.data
        FROM bronze.snapshot_raw AS current
        LEFT JOIN bronze.snapshot_raw AS previous
            ON current.primary_key = previous.primary_key
            AND current.table_name = previous.table_name
            AND previous.ref_date = toDate('{previous_date}')
        WHERE current.ref_date = toDate('{ref_date}')
            AND current.table_name = '{table_name}'
            AND previous.primary_key IS NULL
        """
    )

    client.execute_query(
        f"""
        INSERT INTO silver.delta_events (
            ref_date, table_name, primary_key, operation_type,
            row_hash_before, row_hash_after, data_before, data_after
        )
        SELECT
            toDate('{ref_date}'),
            '{table_name}',
            current.primary_key,
            'UPDATE',
            previous.row_hash,
            current.row_hash,
            previous.data,
            current.data
        FROM bronze.snapshot_raw AS current
        INNER JOIN bronze.snapshot_raw AS previous
            ON current.primary_key = previous.primary_key
            AND current.table_name = previous.table_name
            AND previous.ref_date = toDate('{previous_date}')
        WHERE current.ref_date = toDate('{ref_date}')
            AND current.table_name = '{table_name}'
            AND current.row_hash != previous.row_hash
        """
    )

    client.execute_query(
        f"""
        INSERT INTO silver.delta_events (
            ref_date, table_name, primary_key, operation_type,
            row_hash_before, data_before
        )
        SELECT
            toDate('{ref_date}'),
            '{table_name}',
            previous.primary_key,
            'DELETE',
            previous.row_hash,
            previous.data
        FROM bronze.snapshot_raw AS previous
        LEFT JOIN bronze.snapshot_raw AS current
            ON previous.primary_key = current.primary_key
            AND previous.table_name = current.table_name
            AND current.ref_date = toDate('{ref_date}')
        WHERE previous.ref_date = toDate('{previous_date}')
            AND previous.table_name = '{table_name}'
            AND current.primary_key IS NULL
        """
    )

    client.execute_query(
        f"""
        INSERT INTO silver.current_state (
            table_name, primary_key, row_hash, data,
            first_seen_date, last_seen_date, is_active
        )
        SELECT
            table_name,
            primary_key,
            row_hash,
            data,
            ref_date AS first_seen_date,
            ref_date AS last_seen_date,
            1 AS is_active
        FROM bronze.snapshot_raw
        WHERE ref_date = toDate('{ref_date}')
            AND table_name = '{table_name}'
        """
    )


for config in tables_config:
    process_silver(config["table"])

client.execute_query(
    f"""
    INSERT INTO gold.daily_change_metrics (
        ref_date, table_name, total_inserts, total_updates, total_deletes,
        total_active_records
    )
    SELECT
        ref_date,
        table_name,
        countIf(operation_type = 'INSERT'),
        countIf(operation_type = 'UPDATE'),
        countIf(operation_type = 'DELETE'),
        (
            SELECT count()
            FROM silver.current_state AS cs
            WHERE cs.table_name = de.table_name
                AND cs.is_active = 1
        )
    FROM silver.delta_events AS de
    WHERE ref_date = toDate('{ref_date}')
    GROUP BY ref_date, table_name
    """
)

client.execute_query(
    f"""
    INSERT INTO gold.data_quality_metrics (
        ref_date, table_name, null_count, duplicate_count, total_records,
        quality_score
    )
    SELECT
        toDate('{ref_date}'),
        table_name,
        countIf(data = '' OR data IS NULL),
        count() - uniq(primary_key),
        count(),
        100.0 * (
            1
            - (
                countIf(data = '' OR data IS NULL)
                + count() - uniq(primary_key)
            ) / count()
        ) AS quality_score
    FROM bronze.snapshot_raw
    WHERE ref_date = toDate('{ref_date}')
    GROUP BY table_name
    """
)

summary = client.execute_query_with_result(
    f"""
    SELECT *
    FROM gold.daily_change_metrics
    WHERE ref_date = toDate('{ref_date}')
    """
)
summary.result_rows

client.close()

ModuleNotFoundError: No module named 'pyspark'